In [40]:
import http.client
import boto3
import requests
import pandas as pd
import json
import csv
import logging
import time
# from botocore.exceptions import NoCredentialsError

In [48]:
"""
Get uefa players data
"""

skill_map = {
    1: "goal keepers",
    2: "defenders",
    3: "midfielders",
    4: "attackers"
}

def get_uefa_players_data():
    start_time = time.time()
    logging.info("Fetching players data from UEFA.")
    conn = http.client.HTTPSConnection("gaming.uefa.com")
    conn.request("GET", "/en/uclfantasy/services/feeds/players/players_80_en_13.json")
    res = conn.getresponse()
    data = res.read()
    data = json.loads(data.decode("utf-8"))
    
    cleaned_player_data = []

    # print(data['data']['value']['playerList'][0]('skill'))
    for player in data['data']['value']['playerList']:
        # Transform the skill number to its description
        skill_description = skill_map.get(player.get('skill',0), 'Unknown')

        cleaned_player_data.append({
            'name': player.get('pDName', ''),
            'rating': player.get('rating', ''),
            'value': player.get('value', ''),
            'total points': player.get('totPts', ''),
            'goals': player.get('gS', ''),
            'assist': player.get('assist', ''),
            'minutes played': player.get('minsPlyd', ''),
            'isActive': player.get('isActive', ''),
            'average points': player.get('avgPlayerPts', ''),
            'team': player.get('cCode', ''),
            'man of match': player.get('mOM', ''),
            'position': skill_description,
            'goals conceded': player.get('gC'),
            'yellow cards': player.get('yC'),
            'red cards': player.get('rC'),
            'penalties earned': player.get('pE'),
            'balls recovered': player.get('bR'),
        })
    
    end_time = time.time()
    logging.info(f"UEFA players data fetched and cleaned in {end_time - start_time:.2f} seconds.")
    
    return cleaned_player_data

In [42]:
"""
CSV table creation
"""

def csv_table(player_data):
    # Write to a CSV file
    csv_file_path = 'data/before_quarter_finals_1_leg.csv'

    with open(csv_file_path, 'w', newline='', encoding='utf-8') as csvfile:
        fieldnames = ['name', 'rating', 'value', 'total points', 'goals', 'assist', 'minutes played', 'average points', 'isActive', 'team', 'man of match', 'position', 'goals conceded', 'yellow cards', 'red cards', 'penalties earned', 'balls recovered']
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)

        writer.writeheader()
        for player in player_data:
            writer.writerow(player)

    print("CSV file created successfully.")

In [43]:
"""
CSV
"""
uefa_players_data  = get_uefa_players_data()
csv_table(uefa_players_data)



CSV file created successfully.


In [ ]:
players = get_uefa_players_data()
df = pd.DataFrame(players)
df

,name,rating,value,total points,goals,assist,minutes played,isActive,average points,team,man of match,position,goals conceded,yellow cards,red cards,penalties earned,balls recovered
0,K. Mbappé,0.5,11.1,82,13,1,732,1,9.1,RMA,3,attackers,0,2,0,0,4
1,H. Kane,4.5,10.8,71,10,0,679,1,7.9,BAY,4,attackers,0,0,0,2,10
2,E. Haaland,3.0,10.7,55,8,0,756,0,5.5,MCI,1,attackers,0,0,0,1,6
3,M. Salah,3.5,10.4,45,3,3,629,1,5.0,LIV,0,midfielders,0,0,0,0,12
4,L. Yamal,5.0,9.9,54,5,4,694,1,6.8,BAR,1,midfielders,0,4,0,0,14
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1211,K. Ouattara,1.5,3.8,17,0,0,278,0,2.4,MON,0,defenders,3,1,0,0,8
1212,Eren Elmalı,3.0,3.8,13,0,0,272,0,1.3,GAL,0,defenders,6,0,0,0,14
1213,I. Zabarnyi,3.5,3.8,4,0,0,198,1,0.7,PSG,0,defenders,4,1,1,0,10
1214,M. Garananga,0.0,3.8,3,0,0,44,0,0.8,CPH,0,defenders,2,0,0,0,2


In [52]:
active_df = df[df["isActive"] == 1]
active_df.sort_values("total points", ascending=False)

,name,rating,value,total points,goals,assist,minutes played,isActive,average points,team,man of match,position,goals conceded,yellow cards,red cards,penalties earned,balls recovered
72,D. Szoboszlai,4.0,6.9,83,5,4,885,1,8.3,LIV,2,midfielders,0,1,0,1,47
0,K. Mbappé,0.5,11.1,82,13,1,732,1,9.1,RMA,3,attackers,0,2,0,0,4
24,K. Kvaratskhelia,2.0,8.2,82,7,4,704,1,7.5,PSG,3,midfielders,0,2,0,1,16
46,Vitinha,3.0,7.3,81,6,1,1079,1,6.8,PSG,1,midfielders,0,0,0,0,61
6,Vinícius Júnior,4.5,9.6,78,5,7,990,1,6.5,RMA,2,midfielders,0,2,0,1,10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
983,Javier Navarro,0.0,4.0,0,0,0,0,1,0.0,RMA,0,goal keepers,0,0,0,0,0
995,Á. Pécsi,0.0,4.0,0,0,0,0,1,0.0,LIV,0,goal keepers,0,0,0,0,0
1054,Bruno Ramos,0.0,4.0,0,0,0,0,1,0.0,SPO,0,defenders,0,0,0,0,0
1071,Diego Aguado,0.0,4.0,0,0,0,0,1,0.0,RMA,0,defenders,0,0,0,0,0


In [55]:
points_by_position = df.groupby("position")["total points"].sum()
points_by_position.sort_values(ascending=False)

position
midfielders     7528
defenders       6145
attackers       2836
goal keepers    1189
Name: total points, dtype: int64

In [57]:
players_per_position = df.groupby("position")["name"].count().reset_index()
players_per_position.columns = ["position", "player_count"]
players_per_position.sort_values("player_count", ascending=False)

,position,player_count
3,midfielders,469
1,defenders,401
0,attackers,202
2,goal keepers,144


In [64]:
# Total points by position and team
points_by_pos_team = active_df.groupby(["position", "team"])["total points"].sum().reset_index()

# Filter to just midfielders
midfielders = points_by_pos_team[points_by_pos_team["position"] == "midfielders"]
midfielders.sort_values("total points", ascending=False)

,position,team,total points
30,midfielders,PSG,399
31,midfielders,RMA,374
24,midfielders,ARS,311
28,midfielders,LIV,297
26,midfielders,BAR,252
25,midfielders,ATM,248
27,midfielders,BAY,248
32,midfielders,SPO,224
29,midfielders,MAR,0


In [65]:
attackers = points_by_pos_team[points_by_pos_team["position"] == "attackers"]
attackers.sort_values("total points", ascending=False)

,position,team,total points
1,attackers,ATM,160
3,attackers,BAY,141
2,attackers,BAR,99
6,attackers,RMA,91
0,attackers,ARS,73
4,attackers,LIV,71
7,attackers,SPO,54
5,attackers,PSG,39


In [66]:
defenders = points_by_pos_team[points_by_pos_team["position"] == "defenders"]
defenders.sort_values("total points", ascending=False)

,position,team,total points
8,defenders,ARS,244
13,defenders,PSG,240
12,defenders,LIV,237
15,defenders,SPO,206
14,defenders,RMA,182
11,defenders,BAY,169
9,defenders,ATM,164
10,defenders,BAR,142


In [67]:
keepers = points_by_pos_team[points_by_pos_team["position"] == "goal keepers"]
keepers.sort_values("total points", ascending=False)

,position,team,total points
22,goal keepers,RMA,59
16,goal keepers,ARS,47
20,goal keepers,LIV,40
19,goal keepers,BAY,39
21,goal keepers,PSG,34
23,goal keepers,SPO,31
17,goal keepers,ATM,28
18,goal keepers,BAR,22


In [68]:
top10_by_position = (
    active_df
    .sort_values("total points", ascending=False)
    .groupby("position")
    .head(10)
    [["position", "name", "team", "total points"]]
    .sort_values(["position", "total points"], ascending=[True, False])
)

top10_by_position[top10_by_position["position"] == "midfielders"]

,position,name,team,total points
72,midfielders,D. Szoboszlai,LIV,83
24,midfielders,K. Kvaratskhelia,PSG,82
46,midfielders,Vitinha,PSG,81
6,midfielders,Vinícius Júnior,RMA,78
91,midfielders,Francisco Trincão,SPO,69
81,midfielders,Fermín López,BAR,67
77,midfielders,F. Valverde,RMA,66
33,midfielders,G. Martinelli,ARS,57
22,midfielders,M. Olise,BAY,57
115,midfielders,A. Tchouaméni,RMA,54


In [69]:
top10_by_position = (
    active_df
    .sort_values("total points", ascending=False)
    .groupby("position")
    .head(10)
    [["position", "name", "team", "total points"]]
    .sort_values(["position", "total points"], ascending=[True, False])
)

top10_by_position[top10_by_position["position"] == "attackers"]

,position,name,team,total points
0,attackers,K. Mbappé,RMA,82
12,attackers,J. Alvarez,ATM,75
1,attackers,H. Kane,BAY,71
43,attackers,M. Rashford,BAR,50
500,attackers,Luis Suárez,SPO,44
35,attackers,A. Sørloth,ATM,44
17,attackers,A. Griezmann,ATM,41
14,attackers,V. Gyökeres,ARS,41
36,attackers,Luis Díaz,BAY,38
28,attackers,H. Ekitike,LIV,36


In [70]:
top10_by_position = (
    active_df
    .sort_values("total points", ascending=False)
    .groupby("position")
    .head(10)
    [["position", "name", "team", "total points"]]
    .sort_values(["position", "total points"], ascending=[True, False])
)

top10_by_position[top10_by_position["position"] == "defenders"]

,position,name,team,total points
137,defenders,V. van Dijk,LIV,76
128,defenders,Nuno Mendes,PSG,71
386,defenders,W. Pacho,PSG,66
223,defenders,Gabriel,ARS,50
387,defenders,Marquinhos,PSG,49
194,defenders,A. Hakimi,PSG,46
240,defenders,I. Konaté,LIV,46
746,defenders,D. Hancko,ATM,43
298,defenders,J. Tah,BAY,41
696,defenders,D. Huijsen,RMA,40
